## Colab setup

Run this cell **first**. It clones the repository, installs dependencies,
copies the raw workbook across from Drive, and points the notebooks at the
clone.

**Put your GitHub token in Colab's secrets panel** (the key icon in the left
sidebar), named `GH_TOKEN`, with "Notebook access" switched on. Do not paste
it into a cell — a pasted token gets pushed to GitHub and GitHub will revoke
it automatically.

Safe to re-run. Outside Colab (local Jupyter) it does nothing, so the same
notebook works in both places.

**Colab wipes `/content` when the runtime disconnects.** Push before you
close the tab, or the run is lost — see the last cell of this notebook.


In [ ]:
# ============================================================
# COLAB SETUP — run first. No-op outside Colab. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

GH_USER    = "KT-Devv"
GH_REPO    = "student-dropout-prediction-ghana"
GH_BRANCH  = "main"
DRIVE_XLSX = "/content/drive/MyDrive/Ghana_Dropout_Project/ghana_dropout_study_M.xlsx"

GIT_NAME   = "Your Name"          # <-- edit
GIT_EMAIL  = "your@email"         # <-- edit

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if not IN_COLAB:
    print("Not in Colab — skipping setup. Paths resolve from the repo root.")
else:
    def sh(cmd, check=True):
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        if r.stdout.strip(): print(r.stdout.strip()[:2000])
        if check and r.returncode != 0:
            print(r.stderr.strip()[:2000])
        return r

    # ---- token from the secrets panel, never from a pasted string -------
    TOKEN = None
    try:
        from google.colab import userdata
        TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not TOKEN:
        print("No GH_TOKEN secret found. Cloning read-only — you will be able "
              "to run, but NOT push.\n"
              "Add it: key icon in the left sidebar -> GH_TOKEN -> "
              "Notebook access on.")

    # ---- clone (or reuse an existing clone) -----------------------------
    REPO_PATH = Path(f"/content/{GH_REPO}")
    if REPO_PATH.exists():
        print(f"Repo already present at {REPO_PATH} — pulling latest.")
        sh(f"git -C {REPO_PATH} pull --ff-only", check=False)
    else:
        url = (f"https://{TOKEN}@github.com/{GH_USER}/{GH_REPO}.git" if TOKEN
               else f"https://github.com/{GH_USER}/{GH_REPO}.git")
        r = sh(f"git clone -b {GH_BRANCH} {url} {REPO_PATH}", check=False)
        if not REPO_PATH.exists():
            raise RuntimeError(
                "Clone failed. Check GH_USER/GH_REPO/GH_BRANCH above, and that "
                "your GH_TOKEN has Contents: Read and write on this repository."
            )

    os.chdir(REPO_PATH)
    os.environ["DROPOUT_REPO"] = str(REPO_PATH)
    if str(REPO_PATH) not in sys.path:
        sys.path.insert(0, str(REPO_PATH))

    # ---- dependencies ---------------------------------------------------
    if Path("requirements.txt").exists():
        print("installing requirements (quiet, ~1-2 min on a cold runtime)...")
        sh("pip install -q -r requirements.txt", check=False)

    # ---- raw data: the ONLY thing Drive is used for ---------------------
    # Pupil-level data is never committed (ethics: HuSSREC/AP/543/VOL. 5),
    # so it is copied in at runtime and .gitignore keeps it out of git.
    Path("data-raw").mkdir(exist_ok=True)
    target = Path("data-raw") / Path(DRIVE_XLSX).name
    if target.exists():
        print(f"raw workbook already present: {target}")
    else:
        try:
            from google.colab import drive
            if not os.path.exists("/content/drive/MyDrive"):
                drive.mount("/content/drive")
            if os.path.exists(DRIVE_XLSX):
                sh(f'cp "{DRIVE_XLSX}" data-raw/')
                print(f"copied raw workbook -> {target}")
            else:
                print(f"NOT FOUND: {DRIVE_XLSX}\n"
                      "Fix DRIVE_XLSX above, or upload the workbook to "
                      "data-raw/ manually. Notebooks 2-9 don't need it "
                      "(they read data-processed/cleaned_data.csv).")
        except Exception as e:
            print("Drive mount skipped:", e)

    # ---- git identity, needed before any commit -------------------------
    sh(f'git config user.name "{GIT_NAME}"', check=False)
    sh(f'git config user.email "{GIT_EMAIL}"', check=False)

    # ---- the check worth not skipping -----------------------------------
    r = subprocess.run("git status --porcelain", shell=True, text=True,
                       capture_output=True)
    leaked = [l for l in r.stdout.splitlines()
              if "data-raw" in l or "ghana_dropout_study" in l
              or "cleaned_data.csv" in l]
    if leaked:
        print("\n*** WARNING: pupil-level data is NOT being ignored by git ***")
        for l in leaked: print("   ", l)
        print("Do not commit until .gitignore covers these.")
    else:
        print("\ngit is correctly ignoring the raw data.")

    print(f"\nREPO : {os.getcwd()}")
    print(f"push : {'enabled' if TOKEN else 'DISABLED (no GH_TOKEN)'}")


# Notebook 6 — Hyperparameter Search, Fully Disclosed

## What changed from R01

This notebook is why GATE-2 failed. M13 said *"No automated hyperparameter
search was conducted"*; this notebook ran `RandomizedSearchCV` at n_iter
40 / 40 / 15, and `results/optuna_trials.csv` held 20 more trials whose
generating code was in no committed notebook.

| Change | Reason |
|---|---|
| **Every trial is logged to a committed CSV** | Q26 — undisclosed exploration found by a reviewer after publication is a correction notice |
| Budget is **equal across models**, and stated | R01 gave CatBoost 15 trials at 3 folds while the others got 40 at 5 folds |
| Search scores on **AUC-PR** | R01 searched on F1 while M15 declared AUC-PR primary, so the selected configuration optimised the wrong thing |
| Search runs on the **training pool only** | GATE-1(ii) |
| The orphan artefacts are resolved explicitly | `optuna_trials.csv`, `best_parameters.json`, `best_threshold.txt` — each either regenerated by committed code or declared unused |
| The searched configuration is **not** used for the headline comparison | The research question fixes the configuration by design. The search is reported as exploratory context, not as the reported arm |

**The disclosure this notebook produces is the deliverable**, more than the
tuned model. A search that is reported is normal science.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary)

banner("NOTEBOOK 6 — HYPERPARAMETER SEARCH (DISCLOSED)")
OUT = run_dir("notebook06_tuning")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

N_TRIALS = 40                  # IDENTICAL for every model
TUNE_SEED = SPLIT_SEED
TUNE_FOLDS = 5                 # identical for every model; no per-model override
print(f"budget: {N_TRIALS} trials per model, {TUNE_FOLDS}-fold CV on the "
      f"training pool, scored on {PRIMARY_METRIC}")
print("NO per-model override. R01 gave CatBoost 15 trials at 3 folds.")

In [ ]:
# ---- search space -------------------------------------------------------
from scipy.stats import loguniform, randint, uniform

SPACES = {
    "LightGBM": {
        "learning_rate": loguniform(0.005, 0.3),
        "num_leaves": randint(8, 128),
        "max_depth": randint(3, 20),
        "min_child_samples": randint(5, 40),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "n_estimators": randint(100, 600),
    },
    "RandomForest": {
        "n_estimators": randint(100, 500),
        "max_depth": randint(3, 30),
        "min_samples_split": randint(2, 12),
        "min_samples_leaf": randint(1, 6),
        "max_features": ["sqrt", "log2", None],
    },
}
try:
    from catboost import CatBoostClassifier
    SPACES["CatBoost"] = {
        "depth": randint(3, 10),
        "learning_rate": loguniform(0.01, 0.3),
        "iterations": randint(100, 600),
        "l2_leaf_reg": loguniform(0.5, 10),
    }
except ImportError:
    print("catboost unavailable — record the omission in M13")


def build_estimator(name, params, seed):
    from lightgbm import LGBMClassifier
    from sklearn.ensemble import RandomForestClassifier
    if name == "LightGBM":
        return LGBMClassifier(objective="binary", random_state=seed,
                              verbosity=-1, **params)
    if name == "RandomForest":
        return RandomForestClassifier(random_state=seed,
                                      class_weight="balanced", **params)
    from catboost import CatBoostClassifier
    return CatBoostClassifier(random_state=seed, verbose=0,
                              allow_writing_files=False, **params)

In [ ]:
# ---- the search, with every trial logged -------------------------------
from sklearn.model_selection import ParameterSampler
import time, json

folds = cv_splits(train_pool, TUNE_SEED, n_splits=TUNE_FOLDS, n_repeats=1)
prepared = []
for tr, vl in folds:
    prepared.append(preprocess_inside_fold(train_pool.iloc[tr],
                                           train_pool.iloc[vl]))
print(f"{len(prepared)} folds prepared in-fold\n")

trial_log = []
t0 = time.perf_counter()
for model_name, space in SPACES.items():
    sampler = ParameterSampler(space, n_iter=N_TRIALS, random_state=TUNE_SEED)
    print(f"--- {model_name}: {N_TRIALS} trials ---")
    for ti, params in enumerate(sampler, 1):
        scores = []
        t = time.perf_counter()
        for X_tr, y_tr, X_vl, y_vl, _ in prepared:
            est = build_estimator(model_name, params, TUNE_SEED)
            est.fit(X_tr, y_tr)
            p = est.predict_proba(X_vl)[:, 1]
            scores.append(score_binary(y_vl, p)[PRIMARY_METRIC])
        trial_log.append({
            "model": model_name, "trial": ti,
            "search_algorithm": "RandomizedSearch (ParameterSampler)",
            "scoring_metric": PRIMARY_METRIC,
            "cv_folds": TUNE_FOLDS, "seed": TUNE_SEED,
            "mean_score": float(np.mean(scores)),
            "sd_score": float(np.std(scores, ddof=1)),
            "seconds": time.perf_counter() - t,
            "params_json": json.dumps({k: (int(v) if isinstance(v, np.integer)
                                           else float(v) if isinstance(v, np.floating)
                                           else v) for k, v in params.items()}),
        })
        if ti % 10 == 0:
            print(f"    trial {ti}/{N_TRIALS} "
                  f"best so far {max(r['mean_score'] for r in trial_log if r['model']==model_name):.4f}")
    print(f"  done [{time.perf_counter()-t0:.0f}s]")

trials = pd.DataFrame(trial_log)
trials.to_csv(OUT / "search_trials_ALL.csv", index=False)
print(f"\n{len(trials)} trials logged -> search_trials_ALL.csv")
print("This file IS the M13 disclosure. Commit it.")

In [ ]:
# ---- what M13 must say --------------------------------------------------
disclosure = (trials.groupby("model")
              .agg(trials=("trial", "count"),
                   cv_folds=("cv_folds", "first"),
                   scoring=("scoring_metric", "first"),
                   best_score=("mean_score", "max"),
                   worst_score=("mean_score", "min"),
                   median_score=("mean_score", "median"),
                   total_seconds=("seconds", "sum"))
              .reset_index().sort_values("best_score", ascending=False))
disclosure.to_csv(OUT / "search_disclosure_for_M13.csv", index=False)
print("SEARCH DISCLOSURE TABLE — paste into M13\n")
print(disclosure.round(4).to_string(index=False))

best_rows = trials.loc[trials.groupby("model")["mean_score"].idxmax()]
best = {r["model"]: json.loads(r["params_json"]) for _, r in best_rows.iterrows()}
(OUT / "best_parameters_REGENERATED.json").write_text(json.dumps(best, indent=2))
print("\nbest configuration per model:")
for m, p in best.items():
    print(f"  {m}: {p}")

print("\n" + "!"*72)
print("THESE CONFIGURATIONS ARE NOT USED FOR THE HEADLINE COMPARISON.")
print("The research question fixes the configuration by design (M13), and the")
print("headline arms both use config.SHARED_PARAMS with zero search trials.")
print("This search is reported as exploratory context. Say exactly that in M13.")
print("!"*72)

fig, axes = plt.subplots(1, len(disclosure), figsize=(4*len(disclosure), 3.4),
                         squeeze=False)
for ax, m in zip(axes[0], disclosure["model"]):
    s = trials[trials["model"] == m].sort_values("trial")
    ax.plot(s["trial"], s["mean_score"], ".", alpha=.6)
    ax.plot(s["trial"], s["mean_score"].cummax(), "-", lw=2, label="best so far")
    ax.set_title(m, fontsize=9); ax.set_xlabel("trial")
    ax.set_ylabel(PRIMARY_METRIC); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "figures/search_trajectories.png", dpi=200)
plt.close()

In [ ]:
# ---- resolve the orphan artefacts (Q4, Q26) ----------------------------
# Each of these exists in the repository with no generating code. Removal
# without disclosure is worse than leaving them, so each gets a verdict.
orphans = [
    {"artefact": "results/optuna_trials.csv",
     "status": "SUPERSEDED",
     "action": "Replaced by search_trials_ALL.csv from this notebook, which has "
               "committed generating code. Keep the old file with a README note "
               "recording that its generating code was lost, or delete it and "
               "say so in M13. Do not delete it silently.",
     "used_in_any_reported_result": False},
    {"artefact": "models/best_parameters.json",
     "status": "UNUSED",
     "action": "Regenerated as best_parameters_REGENERATED.json. Neither version "
               "is used for any reported arm. State this in M13.",
     "used_in_any_reported_result": False},
    {"artefact": "models/best_threshold.txt (0.19)",
     "status": "UNUSED — and it contradicts M15",
     "action": "M15 declares the threshold fixed at 0.5 with no optimisation. "
               "Disclose that 0.19 came from an exploratory run, was not used, "
               "and report its caseload consequence in the threshold table "
               "(Notebook 8) so a reviewer can see what it would have meant.",
     "used_in_any_reported_result": False},
    {"artefact": "results/engineered_lightgbm_results.csv (AUC-PR 0.9757)",
     "status": "SUPERSEDED, undocumented change",
     "action": "Records an earlier engineered result against the submitted "
               "0.9789 with nothing naming what changed between them. If the "
               "improvement has no named defect behind it, the instrument's "
               "rule applies: revert and report the original, or name the fix.",
     "used_in_any_reported_result": False},
]
orphan_df = pd.DataFrame(orphans)
orphan_df.to_csv(OUT / "orphan_artefact_resolution.csv", index=False)
for o in orphans:
    print(f"\n[{o['status']}] {o['artefact']}\n  -> {o['action']}")

print("\n\nCOMBINATION COUNT FLOOR for M13 (before folds):")
counted = {
    "baseline classifiers (Notebook 4)": 6,
    "imbalance strategies (Notebook 5)": 5,
    "augmentation configurations (Notebook 5b)": 5,
    "search trials, this notebook": int(len(trials)),
    "legacy Optuna trials (code lost)": 20,
    "ablation grid arms (Notebook 6b)": 9,
    "gamma x alpha cells (Notebook 6b)": 12,
}
for k, v in counted.items():
    print(f"  {k:44s} {v:>4d}")
print(f"  {'TOTAL FLOOR':44s} {sum(counted.values()):>4d}")
pd.DataFrame([counted]).T.rename(columns={0: "count"}).to_csv(
    OUT / "combination_count_floor.csv")

write_manifest(OUT, {"notebook": "06_tuning", "test_set_scored": False,
                     "trials_per_model": N_TRIALS, "cv_folds": TUNE_FOLDS,
                     "scoring_metric": PRIMARY_METRIC,
                     "total_trials": int(len(trials)),
                     "used_for_headline": False,
                     "combination_count_floor": int(sum(counted.values()))})

---

## Save your work

Colab wipes `/content` when the runtime disconnects. Run this before you
close the tab — including `results/`, which has to be committed (the previous
review failed partly because the diagnostic CSVs backing the reported tables
were not in the repository).


In [ ]:
# ---- commit and push this run ----
import os, subprocess, sys
if "google.colab" in sys.modules or os.path.exists("/content"):
    MESSAGE = "Notebook 6 Tuning Disclosed run"     # <-- edit if you like

    def sh(c):
        r = subprocess.run(c, shell=True, text=True, capture_output=True)
        print((r.stdout + r.stderr).strip()[:3000]); return r

    sh("git status --short")
    sh("git add -A")
    sh(f'git commit -m "{MESSAGE}"')
    r = sh("git push")
    if r.returncode != 0:
        print("\nPush failed. Usual causes: no GH_TOKEN secret, or the token "
              "lacks Contents: Read and write. Fix it and re-run this cell — "
              "the commit is already made locally, so nothing is lost until "
              "the runtime disconnects.")
else:
    print("Local run — commit with git as usual.")
